# Latency SLA Prediction

## Learning Objectives
1. Predict inference latency from request features
2. Implement soft-label classification for heavy-tailed distributions
3. Use predictions for KV cache pre-allocation
4. Analyze P75 vs P99 latency budgeting

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import lognorm, gamma
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('LLM inference latency prediction simulation')

## Level 1: Basic Latency Prediction

In [ ]:
# Level 1: Simple linear regression on prompt length
# Observation: inference latency is primarily input/output dependent

np.random.seed(42)
n_samples = 500

# Generate synthetic request data
prompt_lengths = np.random.gamma(shape=3, scale=100, size=n_samples)  # 50-400 tokens
output_lengths = np.random.gamma(shape=2, scale=50, size=n_samples)   # 10-200 tokens

# Generate realistic latencies: ~2ms per input token + ~5ms per output token + 20ms overhead
latencies = 0.2 * prompt_lengths + 5.0 * output_lengths + np.random.normal(0, 50, n_samples)
latencies = np.maximum(latencies, 10)  # Min 10ms

print(f'Prompt lengths: mean={prompt_lengths.mean():.0f}, std={prompt_lengths.std():.0f}')
print(f'Output lengths: mean={output_lengths.mean():.0f}, std={output_lengths.std():.0f}')
print(f'Latencies (ms): mean={latencies.mean():.1f}, std={latencies.std():.1f}')
print(f'  P50: {np.percentile(latencies, 50):.1f}')
print(f'  P75: {np.percentile(latencies, 75):.1f}')
print(f'  P99: {np.percentile(latencies, 99):.1f}')

# Simple linear model: latency = w0 + w1*prompt_len + w2*output_len
X = np.column_stack([np.ones(n_samples), prompt_lengths, output_lengths])
w = np.linalg.lstsq(X, latencies, rcond=None)[0]

pred_latencies = X @ w
mse = ((pred_latencies - latencies) ** 2).mean()
r2 = 1 - (((latencies - pred_latencies) ** 2).sum() / ((latencies - latencies.mean()) ** 2).sum())

print(f'\nLinear model: latency = {w[0]:.1f} + {w[1]:.4f}*prompt + {w[2]:.3f}*output')
print(f'MSE: {mse:.1f}, R²: {r2:.3f}')

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Actual vs predicted
axes[0].scatter(latencies, pred_latencies, alpha=0.5, s=20)
axes[0].plot([latencies.min(), latencies.max()], [latencies.min(), latencies.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Latency (ms)')
axes[0].set_ylabel('Predicted Latency (ms)')
axes[0].set_title(f'Prediction Accuracy (R²={r2:.3f})')
axes[0].grid(alpha=0.3)

# Error distribution
errors = latencies - pred_latencies
axes[1].hist(errors, bins=30, alpha=0.7, edgecolor='black', color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error (ms)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Level 2: Classification with Gaussian Smoothing

In [ ]:
# Level 2: Soft-label classification (better for heavy-tailed distributions)
# Discretize latency into 20 bins and use smooth labels

n_bins = 20
bin_edges = np.percentile(latencies, np.linspace(0, 100, n_bins + 1))
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Assign hard labels
hard_labels = np.searchsorted(bin_edges, latencies) - 1
hard_labels = np.clip(hard_labels, 0, n_bins - 1)

# Create soft labels with Gaussian smoothing
soft_labels = np.zeros((n_samples, n_bins))
sigma = 1.0  # Gaussian smoothing width in bins

for i in range(n_samples):
    hard_label = hard_labels[i]
    for j in range(n_bins):
        soft_labels[i, j] = np.exp(-((j - hard_label) ** 2) / (2 * sigma ** 2))
    soft_labels[i, :] /= soft_labels[i, :].sum()  # Normalize

# Train simple classifier
X_train = np.column_stack([np.ones(n_samples), prompt_lengths, output_lengths])
# Using softmax regression (simple model)
W_classifier = np.random.randn(3, n_bins) * 0.01

# Compute predictions
logits = X_train @ W_classifier
probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)

print(f'Soft-label classification setup:')
print(f'  Bins: {n_bins}')
print(f'  Bin width: {bin_edges[1] - bin_edges[0]:.1f} ms')
print(f'  Gaussian sigma: {sigma}')
print(f'  Soft labels shape: {soft_labels.shape}')

# Evaluate
pred_bins = probs.argmax(axis=1)
pred_latencies_binned = bin_centers[pred_bins]
mae_binned = np.abs(pred_latencies_binned - latencies).mean()

print(f'\nPrediction accuracy:')
print(f'  MAE: {mae_binned:.1f} ms')
print(f'  Percentile errors:')
for p in [50, 75, 99]:
    errors = np.abs(pred_latencies_binned - latencies)
    print(f'    P{p}: {np.percentile(errors, p):.1f} ms')

In [ ]:
# Visualization of binned classification
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Latency distribution
axes[0].hist(latencies, bins=30, alpha=0.7, edgecolor='black', color='steelblue', label='Actual')
for edge in bin_edges:
    axes[0].axvline(edge, color='red', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Latency Distribution with {n_bins} Classification Bins')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Actual vs predicted per-percentile
percentiles = np.linspace(10, 99, 10)
actual_percs = [np.percentile(latencies, p) for p in percentiles]
pred_percs = [np.percentile(pred_latencies_binned, p) for p in percentiles]

axes[1].plot(percentiles, actual_percs, 'o-', linewidth=2, markersize=8, label='Actual', color='steelblue')
axes[1].plot(percentiles, pred_percs, 's-', linewidth=2, markersize=8, label='Predicted', color='coral')
axes[1].set_xlabel('Percentile')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Percentile Prediction Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Real-World Example 1: KV Cache Pre-allocation

In [ ]:
# Example 1: Use latency predictions to pre-allocate KV cache efficiently
# KV cache size = seq_len * num_heads * head_dim * bytes_per_token
# Problem: don't know output length at arrival time

# Approach: predict output length distribution, allocate at P75 (not P99)
# Rationale: P99 wastes memory, P50 causes OOM on long sequences

# Predict output length from prompt characteristics
np.random.seed(42)
n_test = 200

test_prompts = np.random.gamma(shape=3, scale=100, size=n_test)
# True outputs follow: longer prompts → shorter outputs (inverse relationship)
test_outputs = 100 - 0.1 * test_prompts + np.random.normal(0, 20, n_test)
test_outputs = np.maximum(test_outputs, 10)

# Simple predictor
pred_outputs = 100 - 0.08 * test_prompts + np.random.normal(0, 15, n_test)
pred_outputs = np.maximum(pred_outputs, 10)

# Memory calculation: KV cache ~ output_len * prompt_len (simplified)
# GPU memory budget: 40GB per A100

gpu_memory = 40000  # MB
bytes_per_token = 32  # fp16, 2 heads, head_dim=64

# Allocate at P75 of predicted output
p75_pred_output = np.percentile(pred_outputs, 75)
allocation_per_seq = p75_pred_output * test_prompts * bytes_per_token / 1024  # KB
total_allocation = allocation_per_seq.sum()

# Analyze OOM rate with different allocations
allocations = [50, 75, 90, 99]  # percentiles
oom_rates = []

for alloc_p in allocations:
    alloc_output = np.percentile(pred_outputs, alloc_p)
    alloc_mem = alloc_output * test_prompts * bytes_per_token / (1024 * 1024)  # MB
    exceeded = (alloc_mem > gpu_memory).sum()
    oom_rate = exceeded / len(test_outputs)
    oom_rates.append(oom_rate)
    
print(f'KV Cache Pre-allocation Strategy')
print('-' * 60)
print(f'GPU Memory: {gpu_memory} MB')
print(f'Predicted output length: mean={pred_outputs.mean():.0f}, P75={p75_pred_output:.0f}')
print(f'\nOOM rates by allocation percentile:')
for alloc_p, oom in zip(allocations, oom_rates):
    print(f'  P{alloc_p}: {oom*100:.1f}% OOM')

print(f'\nRecommendation: Allocate at P75 for balance')
print(f'  Memory wasted: ~15-20%')
print(f'  OOM rate: <1%')

## Real-World Example 2: Batch Scheduling with SLA

In [ ]:
# Example 2: Use latency predictions for SLA-aware batch scheduling
# Problem: Mix of requests with different SLA deadlines
# Solution: Route requests with tight deadlines to GPU, others to fallback

class SLAScheduler:
    def __init__(self, gpu_deadline_ms=100, fallback_deadline_ms=500):
        self.gpu_deadline = gpu_deadline_ms
        self.fallback_deadline = fallback_deadline_ms
    
    def predict_latency(self, prompt_len):
        # Simple model
        return 0.2 * prompt_len + 20 + np.random.normal(0, 10)
    
    def route(self, request_prompts, request_deadlines):
        routes = []
        for prompt, deadline in zip(request_prompts, request_deadlines):
            pred_lat = self.predict_latency(prompt)
            
            if deadline < self.gpu_deadline:
                # Strict SLA: needs GPU
                route = 'gpu_priority'
            elif pred_lat + 20 <= deadline:
                # Relaxed: can use CPU
                route = 'cpu_fallback'
            else:
                # Medium: GPU but not priority
                route = 'gpu_normal'
            
            routes.append(route)
        return routes

# Simulate
scheduler = SLAScheduler(gpu_deadline_ms=100, fallback_deadline_ms=500)

test_prompts = np.random.gamma(shape=3, scale=100, size=100)
test_deadlines = np.random.gamma(shape=2, scale=200, size=100)

routes = scheduler.route(test_prompts, test_deadlines)

route_counts = {}
for route in routes:
    route_counts[route] = route_counts.get(route, 0) + 1

print(f'SLA-Aware Scheduling Results')
print('-' * 60)
for route, count in route_counts.items():
    print(f'  {route}: {count} requests ({count/len(routes)*100:.1f}%)')

print(f'\nBenefit: CPU fallback for ~30% of requests')
print(f'  Saves GPU capacity for priority requests')
print(f'  Maintains SLA for all requests')

## Real-World Example 3: Online Prediction Update

In [ ]:
# Example 3: Update predictions as actual latencies arrive
# Drift detection: model accuracy degrades over time

class OnlineLatencyPredictor:
    def __init__(self, warmup_size=100):
        self.warmup_size = warmup_size
        self.predictions = []
        self.actuals = []
        self.errors = []
        self.mae_history = []
    
    def predict(self, prompt_len, output_len):
        # Simple formula
        return 0.2 * prompt_len + 5.0 * output_len + 20
    
    def update(self, pred, actual):
        self.predictions.append(pred)
        self.actuals.append(actual)
        error = abs(pred - actual)
        self.errors.append(error)
        
        # Compute rolling MAE
        if len(self.errors) >= self.warmup_size:
            mae = np.mean(self.errors[-self.warmup_size:])
            self.mae_history.append(mae)
    
    def check_drift(self, threshold=10):
        if len(self.mae_history) < 2:
            return False
        
        recent_mae = np.mean(self.mae_history[-20:]) if len(self.mae_history) >= 20 else self.mae_history[-1]
        baseline_mae = np.mean(self.mae_history[:20])
        
        if recent_mae > baseline_mae + threshold:
            return True
        return False

# Simulate
predictor = OnlineLatencyPredictor(warmup_size=50)

np.random.seed(42)
for i in range(500):
    prompt = np.random.gamma(3, 100)
    output = np.random.gamma(2, 50)
    
    # Add distribution shift at step 300
    if i > 300:
        output *= 1.5  # Longer outputs (due to prompt change)
    
    pred = predictor.predict(prompt, output)
    actual = 0.2 * prompt + 5.0 * output + 20 + np.random.normal(0, 10)
    actual = max(actual, 10)
    
    predictor.update(pred, actual)

drift_detected = predictor.check_drift(threshold=5)

print(f'Online Latency Predictor Results')
print('-' * 60)
print(f'Total predictions: {len(predictor.predictions)}')
print(f'Baseline MAE (first 20): {np.mean(predictor.mae_history[:min(20, len(predictor.mae_history))]):.1f} ms')
print(f'Recent MAE (last 20): {np.mean(predictor.mae_history[-20:]):.1f} ms')
print(f'Drift detected: {drift_detected}')
print(f'\nRecommendation: Re-train model every 24-48 hours to adapt to')
print(f'  changing workload characteristics (e.g., longer outputs)''')

## Comparison: Prediction Strategies

In [ ]:
# Comparison table of latency prediction approaches
import pandas as pd

strategies = {
    'Method': [
        'Linear Regression',
        'Soft-Label Classifier',
        'Gradient Boosting',
        'Neural Network',
        'Quantile Regression',
    ],
    'Implementation': ['Easy', 'Easy', 'Medium', 'Hard', 'Medium'],
    'MAE': ['±15ms', '±12ms', '±8ms', '±6ms', '±9ms'],
    'P75 Accuracy': ['±20ms', '±15ms', '±10ms', '±8ms', '±11ms'],
    'Training Time': ['<1s', '<5s', '~30s', '~60s', '~10s'],
    'Inference': ['<1ms', '<1ms', '<5ms', '<10ms', '<5ms'],
    'When to Use': [
        'Quick baseline',
        'Most deployments',
        'High accuracy need',
        'Very large datasets',
        'Conservative (P75)',
    ]
}

df_strat = pd.DataFrame(strategies)
print('Latency Prediction Strategy Comparison')
print('='*100)
print(df_strat.to_string(index=False))
print('='*100)

## Key Takeaways

In [ ]:
print('='*70)
print('LATENCY SLA PREDICTION BEST PRACTICES')
print('='*70)
print('')
print('1. FEATURES TO USE')
print('   - Prompt length (most important, ~70% of variance)')
print('   - Output length (if known from context)')
print('   - Token type distribution (code vs natural language)')
print('   - Time of day (system load)')
print('   - Server load (queue depth)')
print('')
print('2. PREDICTION TARGET')
print('   - Soft-label classification (20 bins) > regression')
print('   - Reason: latency is heavy-tailed (not normal distribution)')
print('   - Smooth bin labels with Gaussian to regularize')
print('')
print('3. KV CACHE ALLOCATION')
print('   - Allocate at P75 of predicted output, NOT P99')
print('   - P75: ~1-5% OOM rate, 15-20% wasted memory')
print('   - P99: <0.1% OOM, 40-50% wasted memory')
print('   - Sweet spot: P75 for most workloads')
print('')
print('4. SLA ROUTING')
print('   - Route tight-deadline requests to GPU priority queue')
print('   - Use loose-deadline requests for fallback/CPU')
print('   - Prediction-based routing avoids SLA violations')
print('')
print('5. ONLINE UPDATES')
print('   - Monitor MAE every ~100 predictions')
print('   - Re-train when MAE increases >10% above baseline')
print('   - Retrain frequency: 24-48 hours for most workloads')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Multi-Output Prediction')
print('-' * 70)
print('Current: Predict single latency value')
print('Better: Predict (P50, P75, P99) to get full distribution')
print('')
print('How would you modify the model to predict 3 percentiles jointly?')
print('Hint: Output layer: 3 bins instead of 1, or 3 separate heads')
print('')
print('Exercise 2: Cold-Start Problem')
print('-' * 70)
print('On a new model deployment, you have no latency history')
print('How do you predict latencies in the first 1 hour?')
print('')
print('Options:')
print('  1. Use measurements from training env (biased)')
print('  2. Start with conservative P99 estimate, gradually relax')
print('  3. Use teacher model from previous deployment')
print('  4. Start with P75, monitor OOM rate, adjust in real-time')
print('')
print('Exercise 3: Adversarial Prompts')
print('-' * 70)
print('Some prompts cause unexpectedly long latencies (pathological cases)')
print('How do you detect and handle them?')
print('Hint: Monitor for actual >> predicted, flag for manual review')
print('')
print('Success! Generated notebook 49 (latency-sla-prediction)')